# Lesson 5: Prompt Engineering

See [docs/05-prompt-engineering.md](../docs/05-prompt-engineering.md).

In [ ]:
import sys, json, random
from functools import partial
sys.path.insert(0, '..')

from src.init_env import set_environment_variables
from src.data_loader import load_mails, split_dataset, build_option_lists
from src.send_request import init_orchestration_service, send_request
from src.evaluation import evalulation_full_dataset, pretty_print_table
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template

set_environment_variables()
init_orchestration_service()

mails = load_mails()
dev_set, test_set, test_set_small = split_dataset(mails)
categories, urgency, sentiment, option_lists = build_option_lists(mails)
mail = dev_set[10]

prompt_8 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.

Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_8 = partial(send_request, prompt=prompt_8, **option_lists)
overall_result = {'basic--llama3.1-70b': evalulation_full_dataset(test_set_small, f_8, categories)}

In [ ]:
random.seed(42)
example_template = '''<example>
{example_input}

## Output

{example_output}
</example>'''
examples = '\n---\n'.join([
    example_template.format(example_input=e['message'], example_output=json.dumps(e['ground_truth']))
    for e in random.sample(dev_set, 3)
])

prompt_10 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Your task is to extract and categorize messages. Here are some example:
{{?few_shot_examples}}
Use the examples when extract and categorize the following message:
Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.'''),
    UserMessage('{{?input}}'),
])
f_10 = partial(send_request, prompt=prompt_10, few_shot_examples=examples, **option_lists)
overall_result['few_shot--llama3.1-70b'] = evalulation_full_dataset(test_set_small, f_10, categories)
pretty_print_table(overall_result)

In [ ]:
example_template_metaprompt = '''<example>
{example_input}

## Output
{key}={example_output}
</example>'''

prompt_get_guide = Template(messages=[
    SystemMessage('''Here are some example:
---
{{?examples}}
---
Use the examples above to come up with a guide on how to distinguish between {{?options}} {{?key}}.
When creating the guide:
- make it step-by-step instructions
- Avoid including explicit information from the examples in the guide
The guide has to cover: {{?options}}'''),
    UserMessage('{{?input}}'),
])

guides = {}
for key in ['categories', 'urgency', 'sentiment']:
    options = option_lists[key]
    selected = '\n---\n'.join([
        example_template_metaprompt.format(example_input=e['message'], key=key, example_output=e['ground_truth'][key])
        for e in dev_set
    ])
    guides[f'guide_{key}'] = send_request(
        prompt=prompt_get_guide,
        examples=selected,
        input=selected,
        key=key,
        options=options,
        _print=False,
        _model='gpt-4o',
    )
print(guides['guide_urgency'][:500])

In [ ]:
prompt_12 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Your task is to classify messages.
This is an explanation of `urgency` labels:
---
{{?guide_urgency}}
---
This is an explanation of `sentiment` labels:
---
{{?guide_sentiment}}
---
This is an explanation of `support` categories:
---
{{?guide_categories}}
---
Giving the following message:
---
{{?input}}
---
extract and return a json with the following keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.'''),
])
f_12 = partial(send_request, prompt=prompt_12, **option_lists, **guides)
overall_result['meta_prompt--llama3.1-70b'] = evalulation_full_dataset(test_set_small, f_12, categories)
pretty_print_table(overall_result)